# 🎤 Arabic Speech-to-Text Model Training
# نظام التعرف على الكلام العربي

Train a custom Arabic STT model using Google Colab's free GPU.

**Runtime**: GPU (T4) - Go to Runtime > Change runtime type > GPU

## Step 1: Check GPU

In [ ]:
# Check if GPU is available
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Mount Google Drive (for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in Drive
!mkdir -p /content/drive/MyDrive/arabic_stt_checkpoints

## Step 3: Clone Project from GitHub

In [ ]:
# ========================================
# Clone from GitHub
# ========================================
!git clone https://github.com/mdIbtisamAnsari/asr-model.git
%cd asr-model

# ========================================
# Alternative: Clone from Google Drive
# ========================================
# If you uploaded the folder to Google Drive instead:
# !cp -r /content/drive/MyDrive/arabic_stt /content/
# %cd /content/arabic_stt

# Verify files are present
!ls -la

In [ ]:
# Verify all required files are present
import os

required_files = ['config.py', 'model.py', 'dataset.py', 'train.py', 'utils.py', 'export_onnx.py']
missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f"❌ Missing files: {missing}")
    print("Please check your git clone or upload these files manually.")
else:
    print("✅ All required files present!")
    !ls -la *.py

## Step 4: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchaudio
!pip install -q librosa soundfile
!pip install -q datasets  # For Common Voice
!pip install -q arabic-reshaper python-bidi
!pip install -q tensorboard
!pip install -q tqdm pandas

print("\n✅ Dependencies installed!")

## Step 5: Configure Training

In [ ]:
# Training configuration
BATCH_SIZE = 16  # Reduce to 8 if you get OOM errors
EPOCHS = 50      # Start with 50, can resume later
LEARNING_RATE = 3e-4

# Save checkpoints to Google Drive (persists after session ends)
CHECKPOINT_DIR = "/content/drive/MyDrive/arabic_stt_checkpoints"
LOG_DIR = "/content/drive/MyDrive/arabic_stt_logs"

!mkdir -p {CHECKPOINT_DIR}
!mkdir -p {LOG_DIR}

print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
print(f"Logs will be saved to: {LOG_DIR}")

## Step 6: Train with Common Voice Dataset

In [ ]:
# This will download Common Voice Arabic (~10GB) and start training
# First run takes longer due to download

# Create symlinks so checkpoints save to Google Drive
!rm -rf checkpoints logs 2>/dev/null
!ln -s {CHECKPOINT_DIR} checkpoints
!ln -s {LOG_DIR} logs

# Start training
!python train.py \
    --use-common-voice \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --lr {LEARNING_RATE} \
    --device cuda

## Alternative: Train with Custom Dataset

In [ ]:
# If you have your own Arabic audio dataset:

# 1. Upload your manifest files (train.tsv, val.tsv)
# from google.colab import files
# uploaded = files.upload()

# 2. Upload audio files to a folder
# !mkdir -p /content/audio
# Upload audio files...

# 3. Train with custom data
# !python train.py \
#     --train-manifest /content/train.tsv \
#     --val-manifest /content/val.tsv \
#     --batch-size 16 \
#     --epochs 100

## Step 7: Monitor Training with TensorBoard

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/arabic_stt_logs

## Step 8: Resume Training (if session disconnected)

In [ ]:
# If your Colab session disconnected, resume from last checkpoint:

!python train.py \
    --use-common-voice \
    --batch-size {BATCH_SIZE} \
    --epochs 100 \
    --resume /content/drive/MyDrive/arabic_stt_checkpoints/best_model.pt

---
# 🎉 Post-Training Steps

## Step 9: Test the Model

In [ ]:
import torch
from model import ArabicSTTModel
from dataset import AudioProcessor

# Load trained model
CHECKPOINT_PATH = "/content/drive/MyDrive/arabic_stt_checkpoints/best_model.pt"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = ArabicSTTModel.load_checkpoint(CHECKPOINT_PATH, device)
model.eval()
print(f"✅ Model loaded on {device}")

In [ ]:
# Upload a test audio file
from google.colab import files
print("Upload an Arabic audio file (WAV/MP3):")
uploaded = files.upload()
test_audio = list(uploaded.keys())[0]
print(f"Uploaded: {test_audio}")

In [ ]:
# Transcribe the audio
audio_processor = AudioProcessor()
features = audio_processor.process_audio(test_audio)
features = features.unsqueeze(0).to(device)

with torch.no_grad():
    transcription = model.decode_greedy(features)

print("\n" + "="*50)
print("📝 Transcription:")
print("="*50)
print(transcription[0])
print("="*50)

## Step 10: Export to ONNX (for lightweight deployment)

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
# Export to ONNX format
!python export_onnx.py \
    --checkpoint /content/drive/MyDrive/arabic_stt_checkpoints/best_model.pt \
    --output /content/drive/MyDrive/arabic_stt_checkpoints/arabic_stt.onnx

print("\n✅ ONNX model exported!")

## Step 11: Download Trained Model

In [ ]:
from google.colab import files

# Download PyTorch model
files.download('/content/drive/MyDrive/arabic_stt_checkpoints/best_model.pt')

# Download ONNX model (smaller, for deployment)
files.download('/content/drive/MyDrive/arabic_stt_checkpoints/arabic_stt.onnx')

## Step 12: Check Model Size

In [ ]:
import os

pt_path = '/content/drive/MyDrive/arabic_stt_checkpoints/best_model.pt'
onnx_path = '/content/drive/MyDrive/arabic_stt_checkpoints/arabic_stt.onnx'

if os.path.exists(pt_path):
    pt_size = os.path.getsize(pt_path) / (1024 * 1024)
    print(f"PyTorch model: {pt_size:.2f} MB")

if os.path.exists(onnx_path):
    onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"ONNX model: {onnx_size:.2f} MB")

---
# 📋 Post-Training Checklist

After training is complete:

1. ✅ **Check metrics**: WER < 30% is good, < 20% is excellent
2. ✅ **Save to Drive**: Checkpoints are auto-saved to Google Drive
3. ✅ **Export ONNX**: For deployment on low-resource devices
4. ✅ **Download models**: `best_model.pt` and `arabic_stt.onnx`
5. ✅ **Test locally**: Run inference on your machine

## Using the Model Locally

```bash
# With PyTorch (needs ~600 MB RAM)
python inference.py --model best_model.pt

# With ONNX (needs ~150 MB RAM)
python inference_lite.py --model arabic_stt.onnx
```

---
# 🔧 Troubleshooting

In [ ]:
# If you get CUDA Out of Memory error:
# 1. Reduce batch size
BATCH_SIZE = 8  # or even 4

# 2. Clear GPU memory
import torch
torch.cuda.empty_cache()

# 3. Restart runtime (Runtime > Restart runtime)

In [ ]:
# If session disconnects frequently:
# Add this to keep session alive (run in separate cell)

import time
from IPython.display import display, Javascript

def keep_alive():
    display(Javascript('console.log("Keeping session alive...")'))

# This runs in background
# keep_alive()

In [ ]:
# Check disk space (Common Voice needs ~15GB)
!df -h /content